In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense

SEED = 88
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.get_logger().setLevel("ERROR")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == \"evaluation\" else Path.cwd()\nDATA_PATH = PROJECT_ROOT / \"datasets/emnist-letters.csv\"\nWEIGHTS_PATH = PROJECT_ROOT / \"models/final_cnn.weights.h5\"

In [4]:
weights_file = Path(WEIGHTS_PATH)

print("Exists:", weights_file.exists())

if weights_file.exists():
    print("Size MB:", round(weights_file.stat().st_size / (1024 * 1024), 2))
    print("Last modified:", datetime.fromtimestamp(weights_file.stat().st_mtime))
else:
    raise FileNotFoundError(f"Could not find weights file: {WEIGHTS_PATH}")

Exists: True
Size MB: 2.83
Last modified: 2026-05-07 14:49:53


In [5]:
# Load and clean dataset
df = pd.read_csv(DATA_PATH, header=None)

expected_labels = range(1, 27)
df_clean = df[df[0].isin(expected_labels)].copy()

X_raw = df_clean.drop(columns=[0]).to_numpy(dtype=np.float32)
y_raw = df_clean[0].to_numpy(dtype=np.int32)

# Same 70/15/15 stratified split
X_train_raw, X_temp_raw, y_train_raw, y_temp_raw = train_test_split(
    X_raw,
    y_raw,
    test_size=0.30,
    random_state=SEED,
    stratify=y_raw
)

X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(
    X_temp_raw,
    y_temp_raw,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp_raw
)

# Same reshape, orientation correction, normalisation, and label encoding
X_val_images = X_val_raw.reshape(-1, 28, 28).transpose(0, 2, 1)[..., np.newaxis] / 255.0
X_test_images = X_test_raw.reshape(-1, 28, 28).transpose(0, 2, 1)[..., np.newaxis] / 255.0

y_val = y_val_raw - 1
y_test = y_test_raw - 1

num_classes = 26

print("Validation shape:", X_val_images.shape, y_val.shape)
print("Test shape:", X_test_images.shape, y_test.shape)

Validation shape: (13320, 28, 28, 1) (13320,)
Test shape: (13320, 28, 28, 1) (13320,)


In [6]:
# Rebuild final selected model architecture
final_model = Sequential([
    Input(shape=(28, 28, 1)),

    Conv2D(32, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),

    Dropout(0.7),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(num_classes, activation="softmax")
], name="final_candidate_weights_verification")

# Load weights before compile
final_model.load_weights(WEIGHTS_PATH)

final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Weights loaded successfully.")
print("Weights file:", WEIGHTS_PATH)
final_model.summary()

Weights loaded successfully.
Weights file: final_cnn.weights.h5


Model: "final_candidate_weights_verification"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 26)             │         3,354 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243,610 (951.60 KB)

 Trainable params: 243,610 (951.60 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Evaluate validation and test metrics
val_loss, val_acc_keras = final_model.evaluate(X_val_images, y_val, verbose=0)
test_loss, test_acc_keras = final_model.evaluate(X_test_images, y_test, verbose=0)

val_proba = final_model.predict(X_val_images, verbose=0)
test_proba = final_model.predict(X_test_images, verbose=0)

val_pred = np.argmax(val_proba, axis=1)
test_pred = np.argmax(test_proba, axis=1)

val_acc = accuracy_score(y_val, val_pred)
val_macro_f1 = f1_score(y_val, val_pred, average="macro")
val_weighted_f1 = f1_score(y_val, val_pred, average="weighted")

test_acc = accuracy_score(y_test, test_pred)
test_macro_f1 = f1_score(y_test, test_pred, average="macro")
test_weighted_f1 = f1_score(y_test, test_pred, average="weighted")

print("Validation metrics from loaded weights:")
print("Validation loss:", round(val_loss, 4))
print("Validation accuracy:", round(val_acc, 4))
print("Validation macro F1:", round(val_macro_f1, 4))
print("Validation weighted F1:", round(val_weighted_f1, 4))

print("\nTest metrics from loaded weights:")
print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_acc, 4))
print("Test macro F1:", round(test_macro_f1, 4))
print("Test weighted F1:", round(test_weighted_f1, 4))

Validation metrics from loaded weights:
Validation loss: 0.1581
Validation accuracy: 0.947
Validation macro F1: 0.9469
Validation weighted F1: 0.9469

Test metrics from loaded weights:
Test loss: 0.145
Test accuracy: 0.9495
Test macro F1: 0.9494
Test weighted F1: 0.9494
